# HPC-PatentTrace 第六步：從爬蟲輸出到釋出資料集

這份 notebook 把本 repository 第五步的輸出 `HPC_15to25_v2.csv` 轉成 HPC-PatentTrace 的釋出資料，也就是論文《具全文、結構化摘要與語意向量的高效能運算專利資料集》的資料集。下面每一節是一個步驟，程式在 `colab/hpc_patenttrace/steps/`，notebook 只顯示各步驟產生的圖與表。

| 節 | 步驟 | 筆數（釋出版本） | GPU |
|---|---|---|---|
| 1 | 設定：套件、參數、檢查、沿用生成結果 | | |
| 2 | 撞字篩選：移除 HPC 指其他意思的紀錄 | 21,947 → 16,434 | 否 |
| 3 | 語料規則：五個段落完整、內文出現展開語 | 16,434 → 7,158 → 6,535 | 否 |
| 4 | 分面：每筆三段短摘要（Llama-3.1-8B-Instruct） | 6,535 | 是 |
| 5 | 摘要層：synthesized、direct-brief、direct-full（Qwen2.5-14B-Instruct） | 6,535 | 是 |
| 6 | SPECTER2 向量、分群與合併主題 | 6,535 | 部分 |
| 7 | 權屬證據與用法示範 | 6,535 | 否 |
| 8 | 摘要層的技術驗證 | 6,535 | 部分 |
| 9 | 釋出檔與計算成本 | | 否 |
| 10 | 執行報告 | | 否 |

**在 Google Colab 執行**

1. 把輸入檔放在 Google Drive 的同一個資料夾（預設 `MyDrive/HPC-PatentTrace/input/`）：`HPC_15to25_v2.csv`、[USPTO Patent Litigation Docket Reports](https://www.uspto.gov/ip-policy/economic-research/research-datasets/patent-litigation-docket-reports-data) 的 `patents.dta`。
2. 選 GPU 執行階段，確認 Parameters 這一格。
3. 全部執行（執行階段 → 全部執行）。輸出寫到同一個資料夾的 `output/full/`。

`REUSE_FROM` 留空時會生成分面與三個摘要層，釋出版本在一張 A100 80 GB 上約花 20 小時。`REUSE_FROM` 指向先前一次完整執行的輸出資料夾時不載入生成模型，其餘步驟全部重新計算，估計用一張 L4 或 A100 約 1 到 1.5 小時。每個耗時的步驟都會寫 checkpoint，中斷後重跑會從停下的地方接著做。

## 參數

填好資料夾與輸入檔即可，notebook 其他地方都不需要改。相對路徑會在 `DATA_DIR` 底下讀取。`sample` 版本隨機抽 400 筆輸入、把每一步都跑一遍；GPU 記憶體小於 32 GB（或沒有 GPU）時，兩個生成步驟都改用小模型 Qwen2.5-0.5B-Instruct，只用來確認程式能跑。

各欄位：`DATA_DIR` 是放輸入與輸出的資料夾；`INPUT_CSV`、`LITIGATION_FILE` 是兩個輸入檔；`REUSE_FROM` 是先前一次完整執行的輸出資料夾（例如 `output/full`，這時本次的 `OUTPUT_DIR` 要換一個；留空就重新生成）；`OUTPUT_DIR` 是輸出資料夾；`PROFILE` 選完整版或抽樣版；`HF_TOKEN` 只有重新生成分面時需要（也可以用名為 HF_TOKEN 的 Colab secret）；`SHOW_LOG` 打勾時把各步驟的紀錄也印在 notebook 裡。

In [ ]:
#@title Parameters
#@markdown **Folder** that holds the input files and receives the outputs (on Colab, a folder of Google Drive).
DATA_DIR = "/content/drive/MyDrive/HPC-PatentTrace"  #@param {type:"string"}
#@markdown **Input files**, relative to DATA_DIR or absolute.
INPUT_CSV = "input/HPC_15to25_v2.csv"  #@param {type:"string"}
LITIGATION_FILE = "input/patents.dta"  #@param {type:"string"}
#@markdown **Reuse** the generated layers of an earlier full run: the output folder of that run, for example `output/full` (this run then needs another OUTPUT_DIR). Leave empty to generate them (about 20 hours on one A100 80 GB).
REUSE_FROM = ""  #@param {type:"string"}
#@markdown **Output** folder, relative to DATA_DIR or absolute; the run writes to OUTPUT_DIR/PROFILE.
OUTPUT_DIR = "output"  #@param {type:"string"}
#@markdown **Profile**: `full` runs the whole input; `sample` runs every step on 400 records drawn at random.
PROFILE = "full"  #@param ["full", "sample"]
SAMPLE_MODELS = "auto"  #@param ["auto", "real", "small"]
#@markdown **Hugging Face token**, needed only to generate the facets with Llama-3.1-8B-Instruct; a Colab secret named HF_TOKEN also works.
HF_TOKEN = ""  #@param {type:"string"}
#@markdown Print the log of each step in the notebook (it is always written to OUTPUT_DIR/PROFILE/logs/).
SHOW_LOG = False  #@param {type:"boolean"}

## 1  設定

1.1 安裝 Colab 沒有的套件，並取得本 repository 的程式（`REPO_REF` 指定分支或標籤）。1.2 掛載 Google Drive、檢查輸入檔與套件；`REUSE_FROM` 指向先前執行的輸出資料夾時，把其中的生成結果複製到本次執行，當作 checkpoint。

寫出：`logs/`、`cell_times.csv`、`runtime_log.csv`、`reuse_manifest.csv`

In [ ]:
#@title 1.1 · Install the packages and get the code of this repository
#@markdown On Google Colab this cell installs the packages and copies the code of the repository; on a local computer, start Jupyter in `colab/` of the repository.
import os, sys
REPO_URL = "https://github.com/Taco-Yun/hpc-patent-dataset"
REPO_REF = "main"  #@param {type:"string"}
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "transformers>=4.45.0" accelerate sentencepiece adapters umap-learn hdbscan bert-score minisom rouge-score sentence-transformers pyarrow
    %pip install -q bitsandbytes
    if not os.path.isdir("/content/hpc-patent-dataset"):
        !git clone -q --depth 1 --branch {REPO_REF} {REPO_URL} /content/hpc-patent-dataset
    sys.path.insert(0, "/content/hpc-patent-dataset/colab")
else:
    sys.path.insert(0, os.getcwd())
from hpc_patenttrace import run_step, show

In [ ]:
#@title 1.2 · Prepare the run
if IN_COLAB and DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
run_step("s1_setup")
show(RUN_SETTINGS)

## 2  撞字篩選

HPC 在運算類專利是高效能運算，在藥學專利是羥丙基纖維素，在渦輪專利是高壓壓縮機。只有紀錄本身的文字顯示其他意思時才移除：作者把縮寫定義成其他全稱，或內文出現其他意思的完整詞組。分類碼只用於報告，不參與判定。

寫出：`HPC_15to25_clean.csv`（候選紀錄）、`provenance/removed_homonym.csv`、`provenance/candidate_ids.csv`、圖 2.1–2.3

In [ ]:
#@title 2 · Homonym screening
run_step("s2_homonym")
show(removed_tab.reset_index(), "Removed records by sense and evidence")

## 3  語料規則

完整性過濾保留標題、摘要、先前技術、詳細說明與請求項都有文字的候選紀錄；收錄規則保留內文出現展開語 `high performance comput*` 的紀錄。通過第一條但沒通過第二條的紀錄，完整收在 `excluded_no_expansion.csv`。

寫出：`released_corpus_ids.csv`、`evidence_flags.csv`、`excluded_no_expansion.csv`、`litigation_family.csv`、`tables/`、圖 3.1

In [ ]:
#@title 3 · Corpus rules
run_step("s3_corpus")
show("tables/funnel_from_crawler_output.csv", "Records at each step")
show("differential_exclusion.csv", "Inclusion rule by class section")

## 4  分面

Llama-3.1-8B-Instruct（4-bit NF4）以貪婪解碼，把每筆紀錄的先前技術（1,500 tokens）、摘要段落（1,500）與詳細說明（2,000）各寫成幾句話。還有缺少的分面時才載入模型。

寫出：`stage1_checkpoint.csv`、`stage1_summaries.csv`、圖 4.1

In [ ]:
#@title 4 · Facets
run_step("s4_facets")

## 5  摘要層

三個摘要層的最後一次呼叫相同：bf16 的 Qwen2.5-14B-Instruct、貪婪解碼、最多 300 個新 token、五到七句。**synthesized** 先做一次推理，再由標題、摘要與分面寫成。

寫出：`stage2_checkpoint.csv`、`patent_full_pipeline.csv`

In [ ]:
#@title 5.1 · Synthesized summaries
run_step("s5_synthesized")

**direct-brief** 與 **direct-full** 以單一提示讀各段落；direct-brief 讀 800、400、800 tokens，direct-full 讀與分面相同的 1,500、1,500、2,000 tokens。

寫出：`baseline_direct.csv`、`baseline_direct_matched.csv`

In [ ]:
#@title 5.2 · direct-brief and direct-full summaries
run_step("s5_direct")

## 6  向量、分群與主題

加上 proximity adapter 的 SPECTER2 把每筆紀錄的標題與 synthesized 摘要編碼成向量。UMAP 降到三維，HDBSCAN 分群（`min_cluster_size=30`、`min_samples=10`，六組掃描中唯一 DBCV 高於 0.3 的設定），再把群合併成 25 個主題。沿用生成結果時，會在釋出的座標上重新分群並與釋出的分群比對，並保留釋出的分群。

寫出：`cluster_results.csv`、`topic_summary.csv`、`merge_map_worksheet.csv`、`hdbscan_param_sweep.csv`、`embeddings/`

In [ ]:
#@title 6 · Embeddings, clusters and topics
run_step("s6_topics")
show("hdbscan_param_sweep.csv", "HDBSCAN settings")
show("topic_summary.csv", "Merged topics")

## 7  權屬證據與用法示範

每件專利的最新讓與紀錄依九條有順序的規則分類；沒有移轉權利的紀錄支持時，`effective_assignee` 保留原始權利人。用法示範包括權利人組合、主題趨勢、引用，以及主題與專利分類的一致程度。

寫出：`ownership_events.csv`、`usage_*.csv`、`topic_classifier_benchmark.csv`、圖 7.1–7.9（7.1、7.2、7.4、7.5 是互動 HTML 檔）

In [ ]:
#@title 7 · Ownership evidence and usage examples
run_step("s7_ownership_usage")
show("usage_assignee_before_after.csv", "Ten largest assignees before and after the ownership evidence")

## 8  技術驗證

每項檢查問的是資料有沒有錯，而不是哪一層比較好：token 預算的涵蓋、保留詞、摘要以外的內容、與錯位摘要對照的配對檢查、人工稽核樣本、由各種表徵做 CPC 與權利人分類、困惑度、與摘要的相似度以及 BERTScore。

寫出：`summary_metrics.csv`、`pairing_integrity_flags.csv`、`added_value_beyond_abstract.csv`、`ipc_downstream_comparison.csv`、`ppl_summaries.csv` 與其他驗證表

In [ ]:
#@title 8 · Technical validation
run_step("s8_validation")
show("summary_metrics.csv", "Summary layers against the abstract")
show("ipc_downstream_comparison.csv", "CPC classification from each representation")

## 9  釋出檔

依本次執行實際使用的物件寫出提示詞、詞表與設定，補齊成本表，並把釋出檔寫到 `release/`。接著 9.4 把 `release/` 所有檔名與欄名中的 ipc 改成 cpc（分類代碼本來就是 CPC），並逐檔與原檔比對：列數、識別碼、分類代碼與其他數值都必須相同。工作檔維持原名。

寫出：`release/`、`release/MANIFEST_release.csv`、`release/NAME_MAPPING_ipc_to_cpc.csv`、`runtime_log_release.csv`

In [ ]:
#@title 9 · Release files
run_step("s9_release")
show("release/MANIFEST_release.csv", "Files of the release", rows=60)
show(NAME_CHECK, "Release names: files converted from ipc to cpc and checked")

## 10  執行報告

報告列出執行環境、各節的實際耗時，完整版另外把本次執行的十七個關鍵數字與釋出執行（2026-10-11）並列。第 2、3 節的筆數由規則決定，必須相同；其餘數字取決於生成結果，沿用釋出執行的生成結果時也相同。

In [ ]:
#@title 10 · Run report
run_step("s10_report")
show(key_df, "Key numbers")
show(by_section.reset_index(), "Wall time by section")
show(env_df, "Environment")

### 附註

- 換了硬體或套件版本重新生成，即使是貪婪解碼，文字也可能不同。
- 分類代碼取自 Google Patents 頁面上的 CPC 路徑。`release/` 裡的名稱一律用 cpc（`main_cpc`、`main_cpc_4`、`main_cpc_3`、`cpc_description`、`cpc_downstream_comparison.csv` 等）；爬蟲輸出與工作檔保留原本含 ipc 的名稱，`release/NAME_MAPPING_ipc_to_cpc.csv` 列出每個新舊名稱的對照。
- `reproduce_figures_tables_tw.ipynb` 由本 notebook 完整執行的輸出資料夾畫出論文的圖與表。